In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings
warnings.filterwarnings('ignore')
#IMPORTSSSSSS
plt.style.use('seaborn-v0_8-darkgrid')
print("Libraries imported successfully!")

## Task #1: Customer Segmentation using All Features

### Objective:
- Perform K-Means clustering using all available features except customer_id
- Compare results with and without feature scaling (exclude age from scaling)
- Analyze differences and insights

In [ ]:
# Task #1: Sample Customer Data
np.random.seed(42)

customer_data = {
    'customer_id': range(1, 31),
    'age': np.random.randint(20, 70, 30),
    'income': np.random.randint(30000, 150000, 30),
    'purchase_frequency': np.random.randint(5, 50, 30),
    'average_purchase_value': np.random.randint(100, 5000, 30),
    'years_as_customer': np.random.randint(1, 20, 30)
}

df_customers = pd.DataFrame(customer_data)
print("Task #1: Customer Data")
print(df_customers.head(10))
print(f"\nDataset shape: {df_customers.shape}")
print(f"\nData types:\n{df_customers.dtypes}")
print(f"\nBasic statistics:\n{df_customers.describe()}")

In [ ]:
# Task #1: K-Means Clustering WITHOUT Scaling
print("\n" + "="*80)
print("Task #1 - PART A: K-Means WITHOUT Feature Scaling")
print("="*80)

# Select features (exclude customer_id)
features_without_id = df_customers.drop('customer_id', axis=1)
print(f"\nFeatures used: {list(features_without_id.columns)}")

# Apply K-Means (k=3 as default)
kmeans_no_scaling = KMeans(n_clusters=3, random_state=42, n_init=10)
df_customers['cluster_no_scaling'] = kmeans_no_scaling.fit_predict(features_without_id)

print(f"\nCluster centers (without scaling):\n{kmeans_no_scaling.cluster_centers_}")
print(f"\nInertia: {kmeans_no_scaling.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(features_without_id, df_customers['cluster_no_scaling']):.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin_score(features_without_id, df_customers['cluster_no_scaling']):.4f}")

print("\nCluster Distribution (without scaling):")
print(df_customers['cluster_no_scaling'].value_counts().sort_index())

print("\nClustered Data (first 15 rows - without scaling):")
print(df_customers[['customer_id', 'age', 'income', 'purchase_frequency', 'average_purchase_value', 'years_as_customer', 'cluster_no_scaling']].head(15))

In [ ]:
# Task #1: K-Means Clustering WITH Feature Scaling (excluding age)
print("\n" + "="*80)
print("Task #1 - PART B: K-Means WITH Feature Scaling (Age excluded)")
print("="*80)

# Create a copy for scaling
features_for_scaling = features_without_id.copy()

# Scale all features except age
scaler = StandardScaler()
features_to_scale = ['income', 'purchase_frequency', 'average_purchase_value', 'years_as_customer']
features_for_scaling[features_to_scale] = scaler.fit_transform(features_for_scaling[features_to_scale])

print(f"\nFeatures after scaling (age kept unscaled):")
print(features_for_scaling.head())

# Apply K-Means with scaled features
kmeans_with_scaling = KMeans(n_clusters=3, random_state=42, n_init=10)
df_customers['cluster_with_scaling'] = kmeans_with_scaling.fit_predict(features_for_scaling)

print(f"\nCluster centers (with scaling):\n{kmeans_with_scaling.cluster_centers_}")
print(f"\nInertia: {kmeans_with_scaling.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(features_for_scaling, df_customers['cluster_with_scaling']):.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin_score(features_for_scaling, df_customers['cluster_with_scaling']):.4f}")

print("\nCluster Distribution (with scaling):")
print(df_customers['cluster_with_scaling'].value_counts().sort_index())

print("\nClustered Data (first 15 rows - with scaling):")
print(df_customers[['customer_id', 'age', 'income', 'purchase_frequency', 'average_purchase_value', 'years_as_customer', 'cluster_with_scaling']].head(15))

In [ ]:
# Task #1: Comparison and Analysis
print("\n" + "="*80)
print("Task #1 - PART C: Comparison and Insights")
print("="*80)

# Compare cluster assignments
comparison = pd.DataFrame({
    'customer_id': df_customers['customer_id'],
    'cluster_no_scaling': df_customers['cluster_no_scaling'],
    'cluster_with_scaling': df_customers['cluster_with_scaling'],
    'clusters_match': df_customers['cluster_no_scaling'] == df_customers['cluster_with_scaling']
})

print("\nComparison of cluster assignments:")
print(comparison.head(15))

matching_customers = comparison['clusters_match'].sum()
total_customers = len(comparison)
match_percentage = (matching_customers / total_customers) * 100

print(f"\n--- KEY INSIGHTS ---")
print(f"Customers in same cluster: {matching_customers}/{total_customers} ({match_percentage:.1f}%)")
print(f"Customers moved to different clusters: {total_customers - matching_customers}/{total_customers} ({100-match_percentage:.1f}%)")

# Analyze differences
print("\n--- IMPACT OF FEATURE SCALING ---")
print("\nWithout Scaling:")
print("- Features with larger ranges (income, purchase_value) dominate the distance calculations")
print("- High-income customers are separated from others even if they have similar behavior patterns")
print("- Clustering is biased towards features with larger numerical values")

print("\nWith Scaling (age excluded):")
print("- All features (except age) have equal influence on clustering")
print("- Features contribute equally to distance metrics")
print("- Better segmentation based on behavioral patterns")
print("- Age remains unscaled as per requirements")

# Create visualization comparing the two approaches
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Without Scaling (using income vs purchase_frequency)
axes[0].scatter(df_customers['income'], df_customers['purchase_frequency'], 
                c=df_customers['cluster_no_scaling'], cmap='viridis', s=100, alpha=0.6)
axes[0].set_xlabel('Income')
axes[0].set_ylabel('Purchase Frequency')
axes[0].set_title('Clustering WITHOUT Feature Scaling')
axes[0].grid(True, alpha=0.3)

# Plot 2: With Scaling
axes[1].scatter(df_customers['income'], df_customers['purchase_frequency'], 
                c=df_customers['cluster_with_scaling'], cmap='viridis', s=100, alpha=0.6)
axes[1].set_xlabel('Income')
axes[1].set_ylabel('Purchase Frequency')
axes[1].set_title('Clustering WITH Feature Scaling (age excluded)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'task1_comparison.png'")

---

## Task #2: Vehicle Fleet Management - K-Means Clustering

### Objective:
- Group vehicles based on usage patterns
- Compare results with and without feature scaling (exclude vehicle_type)
- Analyze clustering effectiveness for fleet management

In [ ]:
# Task #2: Sample Vehicle Data
data = {
    'vehicle_serial_no': [5, 3, 8, 2, 4, 7, 6, 10, 1, 9],
    'mileage': [150000, 120000, 250000, 80000, 100000, 220000, 180000, 300000, 75000, 280000],
    'fuel_efficiency': [15, 18, 10, 22, 20, 12, 16, 8, 24, 9],
    'maintenance_cost': [5000, 4000, 7000, 2000, 3000, 6500, 5500, 8000, 1500, 7500],
    'vehicle_type': ['SUV', 'Sedan', 'Truck', 'Hatchback', 'Sedan', 'Truck', 'SUV', 'Truck', 'Hatchback', 'SUV']
}

df_vehicles = pd.DataFrame(data)
print("\nTask #2: Vehicle Data")
print(df_vehicles)
print(f"\nData types:\n{df_vehicles.dtypes}")
print(f"\nBasic statistics (numeric features only):\n{df_vehicles.describe()}")

In [ ]:
# Task #2: K-Means Clustering WITHOUT Scaling
print("\n" + "="*80)
print("Task #2 - PART A: K-Means WITHOUT Feature Scaling")
print("="*80)

# Select numeric features (exclude vehicle_serial_no and vehicle_type)
features_vehicles_numeric = df_vehicles.drop(['vehicle_serial_no', 'vehicle_type'], axis=1)
print(f"\nFeatures used: {list(features_vehicles_numeric.columns)}")

# Apply K-Means (k=3 as default)
kmeans_vehicles_no_scaling = KMeans(n_clusters=3, random_state=42, n_init=10)
df_vehicles['cluster_no_scaling'] = kmeans_vehicles_no_scaling.fit_predict(features_vehicles_numeric)

print(f"\nCluster centers (without scaling):\n{kmeans_vehicles_no_scaling.cluster_centers_}")
print(f"\nInertia: {kmeans_vehicles_no_scaling.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(features_vehicles_numeric, df_vehicles['cluster_no_scaling']):.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin_score(features_vehicles_numeric, df_vehicles['cluster_no_scaling']):.4f}")

print("\nCluster Distribution (without scaling):")
print(df_vehicles['cluster_no_scaling'].value_counts().sort_index())

print("\nVehicles with cluster assignments (without scaling):")
print(df_vehicles[['vehicle_serial_no', 'mileage', 'fuel_efficiency', 'maintenance_cost', 'vehicle_type', 'cluster_no_scaling']])

In [ ]:
# Task #2: K-Means Clustering WITH Feature Scaling
print("\n" + "="*80)
print("Task #2 - PART B: K-Means WITH Feature Scaling (all numeric features)")
print("="*80)

# Scale all numeric features
scaler_vehicles = StandardScaler()
features_vehicles_scaled = pd.DataFrame(
    scaler_vehicles.fit_transform(features_vehicles_numeric),
    columns=features_vehicles_numeric.columns
)

print(f"\nFeatures after scaling:")
print(features_vehicles_scaled)

# Apply K-Means with scaled features
kmeans_vehicles_with_scaling = KMeans(n_clusters=3, random_state=42, n_init=10)
df_vehicles['cluster_with_scaling'] = kmeans_vehicles_with_scaling.fit_predict(features_vehicles_scaled)

print(f"\nCluster centers (with scaling):\n{kmeans_vehicles_with_scaling.cluster_centers_}")
print(f"\nInertia: {kmeans_vehicles_with_scaling.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(features_vehicles_scaled, df_vehicles['cluster_with_scaling']):.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin_score(features_vehicles_scaled, df_vehicles['cluster_with_scaling']):.4f}")

print("\nCluster Distribution (with scaling):")
print(df_vehicles['cluster_with_scaling'].value_counts().sort_index())

print("\nVehicles with cluster assignments (with scaling):")
print(df_vehicles[['vehicle_serial_no', 'mileage', 'fuel_efficiency', 'maintenance_cost', 'vehicle_type', 'cluster_with_scaling']])

In [ ]:
# Task #2: Comparison and Analysis
print("\n" + "="*80)
print("Task #2 - PART C: Comparison and Fleet Management Insights")
print("="*80)

# Compare cluster assignments
vehicle_comparison = pd.DataFrame({
    'vehicle_serial_no': df_vehicles['vehicle_serial_no'],
    'vehicle_type': df_vehicles['vehicle_type'],
    'cluster_no_scaling': df_vehicles['cluster_no_scaling'],
    'cluster_with_scaling': df_vehicles['cluster_with_scaling'],
    'clusters_match': df_vehicles['cluster_no_scaling'] == df_vehicles['cluster_with_scaling']
})

print("\nComparison of cluster assignments:")
print(vehicle_comparison)

matching_vehicles = vehicle_comparison['clusters_match'].sum()
total_vehicles = len(vehicle_comparison)
match_percentage = (matching_vehicles / total_vehicles) * 100

print(f"\n--- KEY METRICS ---")
print(f"Vehicles in same cluster: {matching_vehicles}/{total_vehicles} ({match_percentage:.1f}%)")
print(f"Vehicles moved to different clusters: {total_vehicles - matching_vehicles}/{total_vehicles} ({100-match_percentage:.1f}%)")

# Analyze by vehicle type
print("\n--- ANALYSIS BY VEHICLE TYPE ---")
print("\nVehicles by type in each cluster (WITHOUT Scaling):")
for cluster in sorted(df_vehicles['cluster_no_scaling'].unique()):
    vehicles_in_cluster = df_vehicles[df_vehicles['cluster_no_scaling'] == cluster]
    print(f"\nCluster {cluster}:")
    print(vehicles_in_cluster[['vehicle_serial_no', 'vehicle_type', 'mileage', 'fuel_efficiency']].to_string())

print("\n\nVehicles by type in each cluster (WITH Scaling):")
for cluster in sorted(df_vehicles['cluster_with_scaling'].unique()):
    vehicles_in_cluster = df_vehicles[df_vehicles['cluster_with_scaling'] == cluster]
    print(f"\nCluster {cluster}:")
    print(vehicles_in_cluster[['vehicle_serial_no', 'vehicle_type', 'mileage', 'fuel_efficiency']].to_string())

print("\n--- FLEET MANAGEMENT INSIGHTS ---")
print("\nWithout Scaling:")
print("- Mileage dominates clustering (ranges from 75K to 300K)")
print("- High-mileage vehicles grouped together regardless of fuel efficiency")
print("- Maintenance costs considered but overshadowed by mileage")
print("- Poor fuel efficiency vehicles not distinguished effectively")

print("\nWith Scaling (Recommended for Fleet Management):")
print("- All factors (mileage, fuel efficiency, maintenance) weighted equally")
print("- Better identification of inefficient vehicles requiring replacement")
print("- Fuel efficiency now has proper influence on segmentation")
print("- Maintenance costs properly considered for fleet planning")
print("- More balanced and actionable clustering for fleet managers")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Without Scaling
scatter1 = axes[0].scatter(df_vehicles['mileage'], df_vehicles['maintenance_cost'], 
                           c=df_vehicles['cluster_no_scaling'], cmap='viridis', s=200, alpha=0.6, edgecolors='black')
axes[0].set_xlabel('Mileage')
axes[0].set_ylabel('Maintenance Cost')
axes[0].set_title('Vehicle Clustering WITHOUT Feature Scaling')
axes[0].grid(True, alpha=0.3)

# Plot 2: With Scaling
scatter2 = axes[1].scatter(df_vehicles['mileage'], df_vehicles['maintenance_cost'], 
                           c=df_vehicles['cluster_with_scaling'], cmap='viridis', s=200, alpha=0.6, edgecolors='black')
axes[1].set_xlabel('Mileage')
axes[1].set_ylabel('Maintenance Cost')
axes[1].set_title('Vehicle Clustering WITH Feature Scaling')
axes[1].grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter1, ax=axes[0])
cbar.set_label('Cluster')
cbar = plt.colorbar(scatter2, ax=axes[1])
cbar.set_label('Cluster')

plt.tight_layout()
plt.savefig('task2_vehicle_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'task2_vehicle_clustering.png'")

---

## Task #3: Student Academic Clustering

### Objective:
- Identify distinct student groups based on academic engagement
- Use Elbow method to determine optimal number of clusters (K: 2-6)
- Visualize clustering results
- Support academic intervention programs

In [ ]:
# Task #3: Sample Student Data
np.random.seed(42)

# Create a realistic student dataset with different performance patterns
n_students = 100

student_data = {
    'student_id': range(1, n_students + 1),
    'GPA': np.random.uniform(1.5, 4.0, n_students),
    'study_hours': np.random.uniform(5, 50, n_students),
    'attendance_rate': np.random.uniform(50, 100, n_students)
}

df_students = pd.DataFrame(student_data)

# Add some structure to create more realistic clusters
# High performers
high_performers = np.random.choice(n_students, 30, replace=False)
df_students.loc[high_performers, 'GPA'] = np.random.uniform(3.5, 4.0, 30)
df_students.loc[high_performers, 'study_hours'] = np.random.uniform(30, 50, 30)
df_students.loc[high_performers, 'attendance_rate'] = np.random.uniform(85, 100, 30)

# Low performers
remaining = np.setdiff1d(range(n_students), high_performers)
low_performers = np.random.choice(remaining, 25, replace=False)
df_students.loc[low_performers, 'GPA'] = np.random.uniform(1.5, 2.5, 25)
df_students.loc[low_performers, 'study_hours'] = np.random.uniform(5, 15, 25)
df_students.loc[low_performers, 'attendance_rate'] = np.random.uniform(50, 70, 25)

print("\nTask #3: Student Data")
print(df_students.head(15))
print(f"\nDataset shape: {df_students.shape}")
print(f"\nBasic statistics:\n{df_students.describe()}")

In [ ]:
# Task #3: Feature Selection and Scaling
print("\n" + "="*80)
print("Task #3 - PART A: Feature Selection and Scaling")
print("="*80)

# Select features for clustering (GPA, study_hours, attendance_rate)
features_students = df_students[['GPA', 'study_hours', 'attendance_rate']]
print(f"\nFeatures selected: {list(features_students.columns)}")

# Apply feature scaling
scaler_students = StandardScaler()
features_students_scaled = pd.DataFrame(
    scaler_students.fit_transform(features_students),
    columns=features_students.columns
)

print(f"\nFeatures before scaling:")
print(features_students.head(10))

print(f"\nFeatures after scaling (StandardScaler):")
print(features_students_scaled.head(10))

print(f"\nMean and Std of scaled features:")
print(f"Mean:\n{features_students_scaled.mean()}")
print(f"Std:\n{features_students_scaled.std()}")

In [ ]:
# Task #3: Elbow Method to Determine Optimal K
print("\n" + "="*80)
print("Task #3 - PART B: Elbow Method (K: 2-6)")
print("="*80)

# Calculate inertia for different K values
inertias = []
silhouette_scores = []
k_values = range(2, 7)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_students_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_students_scaled, kmeans.labels_))

# Display results
print("\nK-Means results for different K values:")
print(f"{'K':<5} {'Inertia':<15} {'Silhouette Score':<20}")
print("-" * 40)
for k, inertia, sil_score in zip(k_values, inertias, silhouette_scores):
    print(f"{k:<5} {inertia:<15.2f} {sil_score:<20.4f}")

# Create Elbow plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method For Optimal K', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(k_values)

# Silhouette score
axes[1].plot(k_values, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(k_values)

plt.tight_layout()
plt.savefig('task3_elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

# Determine optimal K
optimal_k_inertia = k_values[np.argmin(np.diff(inertias))] + 2  # Elbow point heuristic
optimal_k_silhouette = k_values[np.argmax(silhouette_scores)]

print(f"\n--- OPTIMAL K DETERMINATION ---")
print(f"Elbow point (inertia-based): K ≈ {optimal_k_inertia}")
print(f"Best silhouette score: K = {optimal_k_silhouette} (score: {max(silhouette_scores):.4f})")
print(f"\nRecommended K: {optimal_k_silhouette} clusters")
print("\nElbow curve saved as 'task3_elbow_method.png'")

In [ ]:
# Task #3: Perform K-Means with Optimal K
print("\n" + "="*80)
print("Task #3 - PART C: K-Means Clustering with Optimal K")
print("="*80)

# Use optimal K based on silhouette score
optimal_k = optimal_k_silhouette

# Apply K-Means
kmeans_students = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_students['cluster'] = kmeans_students.fit_predict(features_students_scaled)

print(f"\nOptimal K selected: {optimal_k} clusters")
print(f"Inertia: {kmeans_students.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(features_students_scaled, df_students['cluster']):.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin_score(features_students_scaled, df_students['cluster']):.4f}")

# Display cluster distribution
print(f"\nCluster Distribution:")
cluster_counts = df_students['cluster'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(df_students)) * 100
    print(f"Cluster {cluster_id}: {count} students ({percentage:.1f}%)")

# Analyze cluster characteristics
print(f"\n--- Cluster Characteristics ---")
for cluster_id in sorted(df_students['cluster'].unique()):
    cluster_data = df_students[df_students['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id}:")
    print(f"  Average GPA: {cluster_data['GPA'].mean():.2f}")
    print(f"  Average Study Hours: {cluster_data['study_hours'].mean():.2f}")
    print(f"  Average Attendance Rate: {cluster_data['attendance_rate'].mean():.2f}%")
    print(f"  Number of Students: {len(cluster_data)}")

In [ ]:
# Task #3: Visualization of Clusters
print("\n" + "="*80)
print("Task #3 - PART D: Cluster Visualization")
print("="*80)

# Create scatter plot using study_hours and GPA
fig, ax = plt.subplots(figsize=(10, 7))

# Define colors for clusters
colors = plt.cm.Set3(np.linspace(0, 1, optimal_k))

# Plot each cluster
for cluster_id in sorted(df_students['cluster'].unique()):
    cluster_data = df_students[df_students['cluster'] == cluster_id]
    ax.scatter(cluster_data['study_hours'], cluster_data['GPA'], 
               label=f'Cluster {cluster_id}', 
               s=100, 
               alpha=0.6,
               color=colors[cluster_id],
               edgecolors='black',
               linewidth=0.5)

ax.set_xlabel('Average Weekly Study Hours', fontsize=12, fontweight='bold')
ax.set_ylabel('GPA', fontsize=12, fontweight='bold')
ax.set_title(f'Student Clustering Based on Academic Performance (K={optimal_k})', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_student_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

print("Scatter plot saved as 'task3_student_clustering.png'")

In [ ]:
# Task #3: Final Deliverables
print("\n" + "="*80)
print("Task #3 - PART E: Final Deliverables")
print("="*80)

# Display final dataset with cluster assignments
print("\n--- Final Dataset with Cluster Assignments ---")
print("Displaying all students with their clusters:\n")
result_df = df_students[['student_id', 'GPA', 'study_hours', 'attendance_rate', 'cluster']].copy()
result_df = result_df.sort_values('cluster')

# Display in sections
for cluster_id in sorted(df_students['cluster'].unique()):
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id} - Students:")
    print(f"{'='*60}")
    cluster_students = result_df[result_df['cluster'] == cluster_id]
    print(cluster_students.to_string(index=False))

# Create summary report
print("\n\n" + "="*80)
print("ACADEMIC SUPPORT RECOMMENDATIONS BY CLUSTER")
print("="*80)

recommendations = {
    0: "High Performers - Mentorship & Leadership Programs",
    1: "Average Performers - Study Skills Enhancement",
    2: "At-Risk Students - Intensive Tutoring & Counseling",
    3: "Moderate Performers - Peer Learning Groups",
    4: "Struggling Students - Priority Intervention"
}

for cluster_id in sorted(df_students['cluster'].unique()):
    cluster_data = df_students[df_students['cluster'] == cluster_id]
    avg_gpa = cluster_data['GPA'].mean()
    avg_study_hours = cluster_data['study_hours'].mean()
    avg_attendance = cluster_data['attendance_rate'].mean()
    
    print(f"\n📌 CLUSTER {cluster_id}: {recommendations.get(cluster_id, 'Support Program')}")
    print(f"   Students: {len(cluster_data)}")
    print(f"   Avg GPA: {avg_gpa:.2f} | Avg Study Hours: {avg_study_hours:.1f}h/week | Attendance: {avg_attendance:.1f}%")
    
    # Classify cluster
    if avg_gpa >= 3.5:
        status = "✓ Excellent - Consider for tutoring roles"
    elif avg_gpa >= 3.0:
        status = "✓ Good - Maintain current level"
    elif avg_gpa >= 2.5:
        status = "⚠ Satisfactory - Moderate support needed"
    else:
        status = "✗ At Risk - Urgent intervention required"
    
    print(f"   Status: {status}")

# Export results
output_csv = result_df.to_csv(index=False)
print("\n\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Total Students Analyzed: {len(df_students)}")
print(f"Number of Clusters: {optimal_k}")
print(f"Silhouette Score: {silhouette_score(features_students_scaled, df_students['cluster']):.4f}")
print("\nDeliverables Generated:")
print("✓ task3_elbow_method.png - Elbow curve and silhouette analysis")
print("✓ task3_student_clustering.png - Scatter plot visualization")
print("✓ Final dataset with cluster assignments (displayed above)")